# 🔍 Simple RAG with Langfuse Tracing

**Stack:**
- LLM: `gpt-4o-mini` (OpenAI)
- Embeddings: `text-embedding-3-small` (OpenAI)
- Framework: LangChain
- Observability: Langfuse v4 (full tracing of retrieval + generation)
- 🗄️Vector Store: ChromaDB (in-memory)

**What gets traced in Langfuse:**
- Every RAG query (trace)
- Retrieval step (span)
- LLM call with token counts and cost (generation)
- Source documents returned

---
### Before running:
1. Get your **OpenAI API key** → https://platform.openai.com/api-keys
2. Get your **Langfuse keys** → https://cloud.langfuse.com (free tier available)
   - Go to Settings → API Keys → Create new keys
3. Paste them in Cell 2 below

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Langfuse v4 is the latest (as of April 2026)
# langfuse.langchain import path changed in v3+ (was langfuse.callback in v2)
!pip install -q \
    langfuse \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-chroma \
    chromadb \
    openai

print("✅ All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.5/479.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69

In [4]:
# ── Cell 2: 🔑 PASTE YOUR KEYS HERE ──────────────────────────────────────────
# Get Langfuse keys from: https://cloud.langfuse.com → Settings → API Keys
# EU region (default): https://cloud.langfuse.com
# US region:           https://us.cloud.langfuse.com

OPENAI_API_KEY       = "sk-proj---"
LANGFUSE_SECRET_KEY="sk-lf-b346c44a-5e3a-4f7a-aeb9-"
LANGFUSE_PUBLIC_KEY="pk-lf-235de55c-d76e-40c3-ae52-"
LANGFUSE_HOST="https://us.cloud.langfuse.com"

print("🔑 Keys configured. Don't share this notebook with keys inside it!")

🔑 Keys configured. Don't share this notebook with keys inside it!


In [5]:
# ── Cell 3: Initialize Langfuse + OpenAI ──────────────────────────────────────
import os
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler  # v3+ import path (NOT langfuse.callback)

# Set OpenAI key for the LangChain components
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Initialize Langfuse singleton with constructor args (hardcoded, no env vars needed)
Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)

# Get the configured singleton client
langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("✅ Langfuse connected successfully!")
    print(f"   Dashboard → {LANGFUSE_HOST}")
else:
    print("❌ Langfuse auth failed — double-check your keys and host URL")

✅ Langfuse connected successfully!
   Dashboard → https://us.cloud.langfuse.com


In [7]:
# ── Cell 4: Sample Knowledge Base ─────────────────────────────────────────────
# These are the documents your RAG will search through.
# Replace with your own text / load from files / PDFs etc.

from langchain_core.documents import Document

documents = [
    Document(
        page_content="""LangChain is an open-source framework that helps developers build
        applications powered by large language models (LLMs). It provides tools to connect
        models with external data, APIs, and logic. LangChain supports chains, agents,
        memory, and retrieval-augmented generation (RAG) pipelines.""",
        metadata={"source": "langchain_overview", "topic": "frameworks"}
    ),
    Document(
        page_content="""Retrieval-Augmented Generation (RAG) is a technique that combines
        information retrieval with text generation. Instead of relying solely on the LLM's
        training data, RAG fetches relevant documents from a knowledge base at query time
        and uses them as context for the LLM to generate accurate, grounded answers.""",
        metadata={"source": "rag_explanation", "topic": "rag"}
    ),
    Document(
        page_content="""Langfuse is an open-source LLM engineering platform for observability,
        tracing, and evaluation. It captures inputs, outputs, token usage, latency, and costs
        for every LLM call. Langfuse integrates with LangChain via a CallbackHandler that
        automatically instruments your chains without changing your application logic.""",
        metadata={"source": "langfuse_overview", "topic": "observability"}
    ),
    Document(
        page_content="""ChromaDB is an open-source vector database designed for AI applications.
        It stores document embeddings and enables fast semantic similarity search. ChromaDB
        can run in-memory (great for prototyping) or persist to disk for production use cases.""",
        metadata={"source": "chromadb_overview", "topic": "vector_db"}
    ),
    Document(
        page_content="""OpenAI's text-embedding-3-small model produces 1536-dimensional vectors
        and is significantly cheaper and faster than the older ada-002 model, while maintaining
        strong performance on retrieval benchmarks. GPT-4o-mini is a cost-efficient small model
        from OpenAI that delivers strong reasoning at low latency.""",
        metadata={"source": "openai_models", "topic": "models"}
    ),
    Document(
        page_content="""Vector embeddings convert text into high-dimensional numerical vectors
        that capture semantic meaning. Similar texts produce vectors that are close together
        in vector space (measured by cosine similarity or dot product). This enables semantic
        search — finding documents by meaning rather than keyword matching.""",
        metadata={"source": "embeddings_explainer", "topic": "embeddings"}
    ),
]

print(f"📚 Loaded {len(documents)} documents into knowledge base")
for i, doc in enumerate(documents):
    print(f"  [{i+1}] {doc.metadata['source']} (topic: {doc.metadata['topic']})")

📚 Loaded 6 documents into knowledge base
  [1] langchain_overview (topic: frameworks)
  [2] rag_explanation (topic: rag)
  [3] langfuse_overview (topic: observability)
  [4] chromadb_overview (topic: vector_db)
  [5] openai_models (topic: models)
  [6] embeddings_explainer (topic: embeddings)


In [9]:
# ── Cell 5: Create Vector Store with text-embedding-3-small ───────────────────
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding model: text-embedding-3-small
# Cheaper + faster than ada-002, great for RAG
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

# Split documents into chunks (important for longer docs)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
split_docs = text_splitter.split_documents(documents)
print(f"✂️  Split {len(documents)} docs → {len(split_docs)} chunks")

# Build Chroma vector store (in-memory for Colab)
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="rag_demo",
)

# Create retriever — fetches top 3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"✅ Vector store ready with {len(split_docs)} embedded chunks")
print(f"   Embedding model: text-embedding-3-small")

✂️  Split 6 docs → 6 chunks
✅ Vector store ready with 6 embedded chunks
   Embedding model: text-embedding-3-small


In [12]:
# ── Cell 6: Build the RAG Chain with GPT-4o-mini ──────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LLM: gpt-4o-mini — cost-efficient, fast, good reasoning
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,  # deterministic answers for RAG
    openai_api_key=OPENAI_API_KEY,
)

# RAG prompt template
RAG_PROMPT = ChatPromptTemplate.from_template("""\
You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    """Concatenate retrieved documents into a single context string."""
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# Build LangChain LCEL chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built!")
print("   LLM: gpt-4o-mini")
print("   Retriever: ChromaDB (top-3 chunks)")

✅ RAG chain built!
   LLM: gpt-4o-mini
   Retriever: ChromaDB (top-3 chunks)


In [18]:
# ── Cell 7: Run RAG Queries with Langfuse Tracing ─────────────────────────────
# Every call below creates a full trace in Langfuse:
#   Trace → [Retrieval span] → [LLM generation with token counts]

def ask(question: str, session_id: str = "demo-session") -> dict:
    """
    Run a RAG query and trace everything to Langfuse.
    Returns dict with answer + retrieved sources.
    """
    # Create a fresh CallbackHandler per query so traces are separate
    # Pass session_id and user_id for grouping in Langfuse dashboard via config metadata
    langfuse_handler = CallbackHandler(
        # session_id and user_id are now passed via config metadata
        # tags are also passed via config metadata
    )

    # Invoke the RAG chain — pass handler as callback and include session/user IDs in metadata
    answer = rag_chain.invoke(
        question,
        config={
            "callbacks": [langfuse_handler],
            "metadata": {
                "langfuse_session_id": session_id,
                "langfuse_user_id": "colab-user",
                "tags": ",".join(["rag", "gpt-4o-mini", "text-embedding-3-small"])
            }
        }
    )

    # Also retrieve source docs for display (separate call for transparency)
    source_docs = retriever.invoke(question)

    return {
        "question": question,
        "answer": answer,
        "sources": [doc.metadata.get("source") for doc in source_docs],
    }


# ─── Test Queries ───────────────────────────────────────────────────────────
test_questions = [
    "What is RAG and why is it useful?",
    "How does Langfuse integrate with LangChain?",
    "What is text-embedding-3-small and how does it compare to ada-002?",
    "What is ChromaDB used for?",
]

results = []
for i, q in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"❓ Q{i+1}: {q}")
    print("-" * 60)
    result = ask(q, session_id=f"test-session-{i+1}")
    results.append(result)
    print(f"💬 Answer: {result['answer']}")
    print(f"📎 Sources: {result['sources']}")

print(f"\n{'='*60}")
print(f"✅ {len(test_questions)} queries completed and traced to Langfuse!")


❓ Q1: What is RAG and why is it useful?
------------------------------------------------------------
💬 Answer: RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with text generation. It is useful because it fetches relevant documents from a knowledge base at query time and uses them as context for the LLM to generate accurate, grounded answers, rather than relying solely on the LLM's training data.
📎 Sources: ['rag_explanation', 'langchain_overview', 'langfuse_overview']

❓ Q2: How does Langfuse integrate with LangChain?
------------------------------------------------------------
💬 Answer: Langfuse integrates with LangChain via a CallbackHandler that automatically instruments your chains without changing your application logic.
📎 Sources: ['langfuse_overview', 'langchain_overview', 'chromadb_overview']

❓ Q3: What is text-embedding-3-small and how does it compare to ada-002?
------------------------------------------------------------
💬 Answer

In [16]:
# ── Cell 8: ✅ Cross-Verify Retrieved Sources ──────────────────────────────────
# This cell shows EXACTLY what documents were retrieved for each question
# and verifies the answer is grounded in those sources.

print("\n🔎 SOURCE VERIFICATION REPORT")
print("=" * 65)

for i, result in enumerate(results):
    print(f"\nQ{i+1}: {result['question']}")
    print("-" * 65)

    # Retrieve the actual documents with content
    source_docs = retriever.invoke(result["question"])

    for j, doc in enumerate(source_docs):
        src = doc.metadata.get('source', 'unknown')
        topic = doc.metadata.get('topic', 'unknown')
        preview = doc.page_content[:120].replace('\n', ' ').strip()
        print(f"  [{j+1}] 📄 {src} (topic={topic})")
        print(f"       Preview: \"{preview}...\"")

    print(f"  ✅ Answer grounded in: {[d.metadata.get('source') for d in source_docs]}")

print("\n" + "=" * 65)
print("✅ Source verification complete — all answers traced to source docs")


🔎 SOURCE VERIFICATION REPORT

Q1: What is RAG and why is it useful?
-----------------------------------------------------------------
  [1] 📄 rag_explanation (topic=rag)
       Preview: "Retrieval-Augmented Generation (RAG) is a technique that combines         information retrieval with text generation. In..."
  [2] 📄 langchain_overview (topic=frameworks)
       Preview: "LangChain is an open-source framework that helps developers build         applications powered by large language models..."
  [3] 📄 langfuse_overview (topic=observability)
       Preview: "Langfuse is an open-source LLM engineering platform for observability,         tracing, and evaluation. It captures inpu..."
  ✅ Answer grounded in: ['rag_explanation', 'langchain_overview', 'langfuse_overview']

Q2: How does Langfuse integrate with LangChain?
-----------------------------------------------------------------
  [1] 📄 langfuse_overview (topic=observability)
       Preview: "Langfuse is an open-source LLM engineering 

In [17]:
# ── Cell 9: Flush Langfuse (important in short-lived environments like Colab) ──
# Langfuse buffers events and sends them in batches.
# Call flush() to ensure ALL traces are sent before the session ends.

langfuse.flush()
print("✅ All traces flushed to Langfuse!")
print(f"\n🔗 View your traces at: {LANGFUSE_HOST}")
print("   → Go to your project → Tracing → Traces")
print("   → You'll see each query as a separate trace with:")
print("     • Retrieval span (documents fetched)")
print("     • LLM generation (prompt, response, token counts, cost)")
print("     • Latency breakdown for each step")

✅ All traces flushed to Langfuse!

🔗 View your traces at: https://us.cloud.langfuse.com
   → Go to your project → Tracing → Traces
   → You'll see each query as a separate trace with:
     • Retrieval span (documents fetched)
     • LLM generation (prompt, response, token counts, cost)
     • Latency breakdown for each step


In [19]:
# ── Cell 10: Ask Your Own Question (Interactive) ───────────────────────────────
# Change the question below and run this cell as many times as you like.
# Each run creates a new trace in Langfuse.

my_question = "How do embeddings help with semantic search?"

# ── Run it ──
result = ask(my_question, session_id="interactive")

print(f"❓ Question: {result['question']}")
print(f"\n💬 Answer:\n{result['answer']}")
print(f"\n📎 Retrieved from sources: {result['sources']}")

# Flush immediately so you can see it in Langfuse right away
langfuse.flush()
print(f"\n✅ Trace sent → check {LANGFUSE_HOST}")

❓ Question: How do embeddings help with semantic search?

💬 Answer:
Embeddings help with semantic search by converting text into high-dimensional numerical vectors that capture semantic meaning. Similar texts produce vectors that are close together in vector space, allowing for the identification of documents by meaning rather than just keyword matching.

📎 Retrieved from sources: ['embeddings_explainer', 'chromadb_overview', 'openai_models']

✅ Trace sent → check https://us.cloud.langfuse.com


---
## 📊 What You'll See in Langfuse

After running the cells above, go to your Langfuse project dashboard:

| What | Where |
|------|-------|
| All traces | Tracing → Traces |
| Token usage & cost | Inside each trace → LLM generation span |
| Latency per step | Trace timeline view |
| Session grouping | Tracing → Sessions |

Each trace will show:
```
Trace: "RunnableSequence"
  ├── RunnableParallel (retrieval + passthrough)
  │     └── VectorStoreRetriever — returns top-3 chunks
  ├── ChatPromptTemplate
  └── ChatOpenAI (gpt-4o-mini)
        ├── Input tokens: N
        ├── Output tokens: N
        └── Cost: $0.00X
```

## 🔧 Common Issues

| Error | Fix |
|-------|-----|
| `AuthenticationError` | Check your OpenAI API key |
| `Langfuse auth failed` | Check public/secret key + host URL |
| `ModuleNotFoundError: langfuse.callback` | Use `from langfuse.langchain import CallbackHandler` (v3+ path) |
| Empty traces in Langfuse | Make sure you called `langfuse.flush()` |
| Wrong region | EU: `cloud.langfuse.com` · US: `us.cloud.langfuse.com` |